In [1]:
import os
from random import randint
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from pydantic import Field

load_dotenv()

<frozen abc>:106: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.
<frozen abc>:106: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.


True

In [2]:
credential = AzureCliCredential()

from agent_framework.foundry import FoundryAgent

agent = FoundryAgent(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    agent_name="travel-support-agent",
    agent_version="2",
    credential=AzureCliCredential(),
)

In [3]:
@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="The location to get the weather for.")],
) -> str:
    """Get the weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    return f"The weather in {location} is {conditions[randint(0, 3)]} with a high of {randint(30, 38)}°C."

In [4]:
parent_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME"],
    credential=credential,
)

travel_agent_tool = agent.as_tool(
    name="travel-support-agent-tool",
    description=(
        "Authoritative travel knowledge base. MUST be called for ANY question about "
        "hotels, flights, travel packages, destinations, itineraries, tour operators, "
        "or travel agencies (e.g. Margie's Travels). Returns grounded data."
    ),
    arg_name="query",
    arg_description="The user's full travel-related question, passed verbatim.",
)

parent_agent = parent_client.as_agent(
    name="travel-support-agent-parent",
    instructions="""
    You are a travel support assistant parent agent.

    Tool-use policy (STRICT — follow without exception):
    1. If the user's question mentions or relates to hotels, flights, travel packages,
       destinations, itineraries, tour operators, or any travel agency (including
       "Margie's Travels"), you MUST call `travel-support-agent-tool` BEFORE answering.
       Pass the user's question verbatim as the `query` argument.
    2. Do NOT answer travel questions from your own knowledge. Do NOT say a service is
       unknown or has limited online presence — always consult the tool first and base
       your reply only on what the tool returns.
    3. For weather questions, call the `get_weather` tool.
    4. Only respond directly (without tools) for greetings, clarifications, or
       non-travel/non-weather small talk.

    After a tool call, summarize the tool's response for the user.
    """,
    tools=[travel_agent_tool, get_weather],
)

In [5]:
print("Parent Agent (streaming): ", end="", flush=True)

query = "what hotels are offered in las vegas by margie's travels?"

async for chunk in parent_agent.run(query, stream=True):
    if chunk.text:
        print(chunk.text, end="", flush=True)

Parent Agent (streaming): Margie’s Travels offers the following hotels in Las Vegas:

1. **The Volcano Hotel**: Located on The Strip, featuring a casino, live entertainment, and a beautiful pool area.
2. **The Fountain Hotel**: A luxury hotel with fine dining restaurants and cocktail bars.
3. **The Canal Hotel**: An Italian-themed resort with opulent suites.

You can find more details or book your stay directly on their website at [www.margiestravel.com](http://www.margiestravel.com). Let me know if you'd like assistance with anything else!

In [6]:
print("Parent Agent (streaming): ", end="", flush=True)

query = "what's the weather like in las vegas?"

async for chunk in parent_agent.run(query, stream=True):
    if chunk.text:
        print(chunk.text, end="", flush=True)

Parent Agent (streaming): The weather in Las Vegas is currently rainy, with a high of 36°C.